In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
Larr = np.logspace(1,4,10)

colors = plt.cm.rainbow(np.linspace(0,1,len(Larr)))
ylabels = [r"Order Parameter $s_{max}/N$",r"Total Number of Clusters, Normalized $\sum n_s/N$",r"Susceptiblity $\chi$"]
for i in range(3):
    fig1,ax1 = plt.subplots()
    fig2,ax2 = plt.subplots()
    for j in range(len(Larr)):
        L= int(Larr[j])
        filelist = Path("./postprocess_C7").glob(f"postprocessedquantities_L_{L}_runs_*.npy")
        # print(filelist)
        for file in filelist:
            # print(file)
            arr = np.load(file,mmap_mode='r')
            # print(arr)
            step = max(1,(L*L)//1000)
            if i == 1:
                ax1.plot(np.arange(1,L*L+1,step)/(L*L),arr[::step,i,0]/(L*L),color=colors[j],label=r"$L={}$".format(L))
            else:
                ax1.plot(np.arange(1,L*L+1,step)/(L*L),arr[::step,i,0],color=colors[j],label=r"$L={}$".format(L))
            ax2.plot(np.arange(1,L*L+1,step)/(L*L),arr[::step,i,1]/arr[::step,i,0],color=colors[j],label=r"$L={}$".format(L))
    ax1.set_xlabel(r"Occupation $n/N$")
    ax1.set_ylabel(ylabels[i])
    ax1.set_title(ylabels[i])
    ax2.set_xlabel(r"Occupation $n/N$")
    ax2.set_ylabel("Fractional Error in "+ylabels[i])
    ax2.set_title("Fractional Error in "+ylabels[i])
    if i == 2:
        ax1.set_yscale("log")
    ax1.legend()
    ax2.legend()    
    ax2.set_yscale("log")    
    plt.show()
    plt.show()


In [ ]:
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt

def nz_binomial_transform(Qn, Qnerr, pvals, nsigma=8):
    N = len(Qn)
    Qp = np.zeros(len(pvals))
    Qperr = np.zeros(len(pvals))

    for i in range(len(pvals)):
        p = pvals[i]
        if p == 0:
            Qp[i] = Qn[0]
            continue
        if p == 1:
            Qp[i] = Qn[N-1]
            continue
        mean = N * p  
        sigma = np.sqrt(N * p * (1 - p))
        nmin = max(0, int(mean - nsigma * sigma)-1)#Avoid summing over entire dataset, only 8 sigma closest to mean
        nmax = min(N, int(mean + nsigma * sigma) )#remember python indexing, mean is at index mean-1
        n = np.arange(nmin, nmax)
        weights = binom.pmf(n+1, N, p)
        Qp[i] = np.nansum(Qn[n] * weights)
        Qperr[i] = np.sqrt(np.nansum(np.square(Qnerr[n]*weights)))
    return Qp,Qperr
Larr = np.logspace(1,4,10)
pvals = np.linspace(0.1,0.9,1000)
colors = plt.cm.rainbow(np.linspace(0,1,len(Larr)))
ylabels = [r"Order Parameter $s_{max}/N$",r"Total Number of Clusters, Normalized $\sum n_s/N$",r"Susceptiblity $\chi$"]
for i in range(3):
    fig1,ax1 = plt.subplots()
    fig2,ax2 = plt.subplots()
    for j in range(len(Larr)):
        L= int(Larr[j])
        print(i,j)
        filelist = Path("./postprocess_C7").glob(f"postprocessedquantities_L_{L}_runs_*.npy")
        # print(filelist)
        for file in filelist:
            # print(file)
            arr = np.load(file,mmap_mode='r')
            # print(arr)
            if i == 1:
                qp,qperr = nz_binomial_transform(arr[:,i,0],arr[:,i,1],pvals)
                qp, qperr = qp / (L*L), qperr / (L*L)

                ax1.plot(pvals,qp,color=colors[j],label=r"$L={}$".format(L))
            else:
                qp,qperr = nz_binomial_transform(arr[:,i,0],arr[:,i,1],pvals)

                ax1.plot(pvals,qp,color=colors[j],label=r"$L={}$".format(L))
            ax2.plot(pvals,qperr/qp,color=colors[j],label=r"$L={}$".format(L))
    ax1.set_xlabel(r"Occupation $p$")
    ax1.set_ylabel(ylabels[i])
    ax1.set_title(ylabels[i])
    ax2.set_xlabel(r"Occupation Probability $p$")
    ax2.set_ylabel("Fractional Error in "+ylabels[i])
    ax2.set_title("Fractional Error in "+ylabels[i])
    if i == 2:
        ax1.set_yscale("log")
    ax1.legend()
    ax2.legend()    
    ax2.set_yscale("log")    
    plt.show()
    plt.show()
    

In [ ]:
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt
from pathlib import Path
def nz_binomial_transform(Qn, Qnerr, pvals, nsigma=5):
    N = len(Qn)
    Qp = np.zeros(len(pvals))
    Qperr = np.zeros(len(pvals))

    for i in range(len(pvals)):
        p = pvals[i]
        if p == 0:
            Qp[i] = Qn[0]
            continue
        if p == 1:
            Qp[i] = Qn[N-1]
            continue
        mean = N * p  
        sigma = np.sqrt(N * p * (1 - p))
        nmin = max(0, int(mean - nsigma * sigma)-1)#Avoid summing over entire dataset, only 8 sigma closest to mean
        nmax = min(N, int(mean + nsigma * sigma) )#remember python indexing, mean is at index mean-1
        n = np.arange(nmin, nmax)
        weights = binom.pmf(n+1, N, p)
        Qp[i] = np.nansum(Qn[n] * weights)
        Qperr[i] = np.sqrt(np.nansum(np.square(Qnerr[n]*weights)))
    return Qp,Qperr
Larr = np.logspace(1,4,10)
pvals = np.linspace(0.3,0.7,3000)
colors = plt.cm.rainbow(np.linspace(0,1,len(Larr)))
ylabels = [r"Order Parameter $s_{max}/N$",r"Total Number of Clusters, Normalized $\sum n_s/N$",r"Susceptiblity $\chi$"]
Warr = np.zeros((Larr.shape[0],pvals.shape[0]))
for i in range(1):
    fig1,ax1 = plt.subplots()
    fig2,ax2 = plt.subplots()
    for j in range(len(Larr)):
        L= int(Larr[j])
        print(i,j)
        filelist = Path("./postprocess_C7").glob(f"postprocessedquantities_L_{L}_runs_*.npy")
        # print(filelist)
        for file in filelist:
            # print(file)
            arr = np.load(file,mmap_mode='r')
            # print(arr)
            if i == 1:
                qp,qperr = nz_binomial_transform(arr[:,i,0],arr[:,i,1],pvals)
                qp, qperr = qp / (L*L), qperr / (L*L)

                ax1.plot(pvals,qp,color=colors[j],label=r"$L={}$".format(L))
            else:
                qp,qperr = nz_binomial_transform(arr[:,i,0],arr[:,i,1],pvals)

                ax1.plot(pvals,qp,color=colors[j],label=r"$L={}$".format(L))
            Warr[j,:] = qp

            ax2.plot(pvals,qperr/qp,color=colors[j],label=r"$L={}$".format(L))
    ax1.set_xlabel(r"Occupation $p$")
    ax1.set_ylabel(ylabels[i])
    ax1.set_title(ylabels[i])
    ax2.set_xlabel(r"Occupation Probability $p$")
    ax2.set_ylabel("Fractional Error in "+ylabels[i])
    ax2.set_title("Fractional Error in "+ylabels[i])
    if i == 2:
        ax1.set_yscale("log")
    ax1.legend()
    ax2.legend()    
    ax2.set_yscale("log")    
    plt.show()
    plt.show()
    

In [ ]:
# finite size scaling
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
#Larr
#Order parameter in Warr[L,p]
#pvals

xrange  = 0.6
xbase = np.linspace(-xrange,xrange,10000)
def finite_size_scaling_residuals(xbase,pc,beta,nu):
    mean = np.zeros(len(xbase))
    residuals = np.zeros(len(xbase))
    for i in range(len(Larr)):
        L = Larr[i]
        x = (pvals-pc)*L**(1/nu)
        y = Warr[i,:]*L**(beta/nu)
        f = interp1d(
            x,
            y,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        # print(x)
        # print(y)
        regridy = f(xbase)
        mean += regridy
    mean /= len(Larr)
    for i in range(len(Larr)):
        L = Larr[i]
        x = (pvals-pc)*L**(1/nu)
        y = Warr[i,:]*L**(beta/nu)
        f = interp1d(
            x,
            y,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        regridy = f(xbase)
    residuals += np.square(mean-regridy)
    return residuals
p0 = [0.5926,5/36,4/3]
popt,pcov = curve_fit(finite_size_scaling_residuals,xbase,np.zeros(xbase.shape[0]),p0=p0)
pc,beta,nu =popt
perr = np.sqrt(np.diag(pcov))
print(popt)
print(perr)
for i in range(len(Larr)):
    x = (pvals-pc)*Larr[i]**(1/nu)
    y = Warr[i,:]*(Larr[i]**(beta/nu))
    plt.plot(x,y,label=f"L={Larr[i]}")
plt.xlim(np.min(xbase),np.max(xbase))
plt.xlabel(r"$(p_i-p_c)L^{1/\nu}$")
plt.ylabel(r"$P_iL^{\beta/\nu}$")
plt.title("Data Collapse of Order Parameter $P$")
plt.show()
plt.plot(xbase,finite_size_scaling_residuals(xbase,*popt))
plt.title("Variance of Best Fit Data Collapse From Mean")
plt.show()

In [ ]:
# finite size scaling
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
#Larr
#Order parameter in Warr[L,p]
#pvals
pvals = np.linspace(0.3,0.7,3000)

Warr = np.zeros((len(Larr), len(pvals)))
for j, size in enumerate(Larr):
    data_file = next(Path("./postprocess_C7").glob(f"postprocessedquantities_L_{int(size)}_runs_*.npy"))
    arr = np.load(data_file, mmap_mode="r")
    Warr[j], _ = nz_binomial_transform(arr[:, 2, 0], arr[:, 2, 1], pvals)

xrange  = 0.6
xbase = np.linspace(-xrange,xrange,10000)
def finite_size_scaling_residuals(xbase,pc,gamma,nu):
    mean = np.zeros(len(xbase))
    residuals = np.zeros(len(xbase))
    for i in range(len(Larr)):
        L = Larr[i]
        x = (pvals-pc)*L**(1/nu)
        y = Warr[i,:]*L**(-gamma/nu)
        f = interp1d(
            x,
            y,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        # print(x)
        # print(y)
        regridy = f(xbase)
        mean += regridy
    mean /= len(Larr)
    for i in range(len(Larr)):
        L = Larr[i]
        x = (pvals-pc)*L**(1/nu)
        y = Warr[i,:]*L**(-gamma/nu)
        f = interp1d(
            x,
            y,
            kind="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        regridy = f(xbase)
    residuals += np.square(mean-regridy)
    return residuals
p0 = [0.5926,2.389,4/3]
popt,pcov = curve_fit(finite_size_scaling_residuals,xbase,np.zeros(xbase.shape[0]),p0=p0)
pc,beta,nu =popt
perr = np.sqrt(np.diag(pcov))
print(popt)
print(perr)
for i in range(len(Larr)):
    x = (pvals-pc)*Larr[i]**(1/nu)
    y = Warr[i,:]*(Larr[i]**(-beta/nu))
    plt.plot(x,y,label=f"L={Larr[i]}")
plt.xlim(np.min(xbase),np.max(xbase))
plt.xlabel(r"$(p_i-p_c)L^{1/\nu}$")
plt.ylabel(r"$\chi_iL^{-\gamma/\nu}$")
plt.title(r"Data Collapse of Susceptibility $\chi$")
plt.show()
plt.title(r"Variance of Best Fit Data Collapse From Mean")

plt.plot(xbase,finite_size_scaling_residuals(xbase,*popt))
plt.show()

In [ ]:
#susceptibility maximum
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt
from pathlib import Path

def nz_binomial_transform(Qn, Qnerr, pvals, nsigma=5):
    N = len(Qn)
    Qp = np.zeros(len(pvals))
    Qperr = np.zeros(len(pvals))

    for i in range(len(pvals)):
        p = pvals[i]
        if p == 0:
            Qp[i] = Qn[0]
            continue
        if p == 1:
            Qp[i] = Qn[N-1]
            continue
        mean = N * p  
        sigma = np.sqrt(N * p * (1 - p))
        nmin = max(0, int(mean - nsigma * sigma)-1)#Avoid summing over entire dataset, only 8 sigma closest to mean
        nmax = min(N, int(mean + nsigma * sigma) )#remember python indexing, mean is at index mean-1
        n = np.arange(nmin, nmax)
        weights = binom.pmf(n+1, N, p)
        Qp[i] = np.nansum(Qn[n] * weights)
        Qperr[i] = np.sqrt(np.nansum(np.square(Qnerr[n]*weights)))
    return Qp,Qperr
Larr = np.logspace(1,4,10)
maxxi = np.zeros(Larr.shape[0])
maxxierr = np.zeros(Larr.shape[0])
maxchierr = np.zeros(Larr.shape[0])
maxchi =  np.zeros(Larr.shape[0])
pvals = np.linspace(0.45,0.61,1000)
colors = plt.cm.rainbow(np.linspace(0,1,len(Larr)))
ylabels = [r"Order Parameter $s_{max}/N$",r"Total Number of Clusters, Normalized $\sum n_s/N$",r"Susceptiblity $\chi$"]
for i in range(2,3):
    fig1,ax1 = plt.subplots()
    fig2,ax2 = plt.subplots()
    for j in range(len(Larr)):
        L= int(Larr[j])
        print(i,j)
        filelist = Path("./postprocess_C7").glob(f"postprocessedquantities_L_{L}_runs_*.npy")
        # print(filelist)
        for file in filelist:
            # print(file)
            arr = np.load(file,mmap_mode='r')
            # print(arr)
    
            qp,qperr = nz_binomial_transform(arr[:,i,0],arr[:,i,1],pvals)
            index = np.argmax(qp)
            maxxi[j] = pvals[index]
            slope = (qp[index]-qp[index-1])/(pvals[index]-pvals[index-1])
            maxxierr[j] = qperr[index]/slope
            maxchi[j] = qp[index]
            maxchierr[j] = qperr[index]

            ax1.plot(pvals,qp,color=colors[j],label=r"$L={}$".format(L))
            ax2.plot(pvals,qperr/qp,color=colors[j],label=r"$L={}$".format(L))
    ax1.set_xlabel(r"Occupation $p$")
    ax1.set_ylabel(ylabels[i])
    ax1.set_title(ylabels[i])
    ax2.set_xlabel(r"Occupation Probability $p$")
    ax2.set_ylabel("Fractional Error in "+ylabels[i])
    ax2.set_title("Fractional Error in "+ylabels[i])
    if i == 2:
        ax1.set_yscale("log")
    ax1.legend()
    ax2.legend()    
    ax2.set_yscale("log")    
    plt.show()
    plt.show()
    

In [ ]:
print(maxxi)
from scipy.optimize import curve_fit
def expdecay(x,pc,nu,K):
    return pc+K*(x**(-1/nu))
Lfit = Larr
maxxifit = maxxi
popt,pcov = curve_fit(expdecay,Lfit,maxxifit)
print(popt)
print(np.sqrt(np.diag(pcov)))
for j in range(len(Larr)):
    plt.errorbar(Larr[j],maxxi[j],yerr=1e1*maxxierr[j],fmt="o",color="blue")
    if j ==2:
        plt.errorbar(Larr[j],maxxi[j],yerr=1e1*maxxierr[j],fmt="o",color="blue",label=r"Data, Errors $\times 10$")

xarr = np.linspace(45,2200)
plt.title("Finite Size Scaling of Critical Point Susceptibility")
plt.plot(xarr,expdecay(xarr,*popt))
plt.hlines(popt[0],xmin=45,xmax=2156,color="black",label=rf"$p_c =${popt[0]:.4f} $\pm${np.sqrt(np.diag(pcov))[0]:.4f}")
plt.xlabel("L")
plt.ylabel(r"Occupation Probability of Maximum $\chi$,$p_{\chi, max}$")
plt.legend()
# plt.xscale("log")
# plt.yscale("log")

In [ ]:
# batch12 peak susceptibility scaling

print(maxchi)
from scipy.optimize import curve_fit
def expdecay(x,gammanu,K):
    return K*x**gammanu
popt,pcov = curve_fit(expdecay,Larr[2:-2],maxchi[2:-2])
print(popt)
print(np.sqrt(np.diag(pcov)))
for j in range(2,len(Larr)-2):
    plt.errorbar(Larr[j],maxchi[j],yerr=1e3*maxchierr[j],fmt="o",color="blue")
    if j == 2:
        plt.errorbar(Larr[j],maxchi[j],yerr=1e3*maxchierr[j],fmt="o",color="blue",label=r"Data, Errors $\times 1000$")

xarr = np.linspace(45,2200)
plt.title(r"Finite Size Scaling of Peak Susceptibility $\xi_{max}$")
plt.plot(xarr,expdecay(xarr,*popt),label=fr"$\gamma/\nu$ = {popt[0]:.4f}$\pm${np.sqrt(np.diag(pcov))[0]:.4f}")
# plt.hlines(popt[0],xmin=45,xmax=2156,color="black",label=r"$p_c$")
plt.xlabel("L")
plt.ylabel(r"$\xi_{max}$")
plt.xscale("log")
plt.yscale("log")
plt.legend()

In [ ]:
# batch13 order parameter scaling at pc
pc = 0.59275
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt
from pathlib import Path
def nz_binomial_transform(Qn, Qnerr, pvals, nsigma=5):
    N = len(Qn)
    Qp = np.zeros(len(pvals))
    Qperr = np.zeros(len(pvals))

    for i in range(len(pvals)):
        p = pvals[i]
        if p == 0:
            Qp[i] = Qn[0]
            continue
        if p == 1:
            Qp[i] = Qn[N-1]
            continue
        mean = N * p  
        sigma = np.sqrt(N * p * (1 - p))
        nmin = max(0, int(mean - nsigma * sigma)-1)#Avoid summing over entire dataset, only 8 sigma closest to mean
        nmax = min(N, int(mean + nsigma * sigma) )#remember python indexing, mean is at index mean-1
        n = np.arange(nmin, nmax)
        weights = binom.pmf(n+1, N, p)
        Qp[i] = np.nansum(Qn[n] * weights)
        Qperr[i] = np.sqrt(np.nansum(np.square(Qnerr[n]*weights)))
    return Qp,Qperr
Larr = np.logspace(1,4,10)
pvals = np.array([pc])
colors = plt.cm.rainbow(np.linspace(0,1,len(Larr)))
ylabels = [r"Order Parameter $s_{max}/N$",r"Total Number of Clusters, Normalized $\sum n_s/N$",r"Susceptiblity $\chi$"]
Warr = np.zeros((Larr.shape[0],2))
for i in range(1):
    fig1,ax1 = plt.subplots()
    fig2,ax2 = plt.subplots()
    for j in range(len(Larr)):
        L= int(Larr[j])
        print(i,j)
        filelist = Path("./postprocess_C7").glob(f"postprocessedquantities_L_{L}_runs_*.npy")
        # print(filelist)
        for file in filelist:
            # print(file)
            arr = np.load(file,mmap_mode='r')
            
            qp,qperr = nz_binomial_transform(arr[:,i,0],arr[:,i,1],pvals)
            print(qp)
            ax1.plot(pvals,qp,marker="o",color=colors[j],label=r"$L={}$".format(L))
            Warr[j,0] = qp[0]
            Warr[j,1] = qperr[0]

            ax2.plot(pvals,qperr/qp,marker="o",color=colors[j],label=r"$L={}$".format(L))
    ax1.set_xlabel(r"Occupation $p$")
    ax1.set_ylabel(ylabels[i])
    ax1.set_title(ylabels[i])
    ax2.set_xlabel(r"Occupation Probability $p$")
    ax2.set_ylabel("Fractional Error in "+ylabels[i])
    ax2.set_title("Fractional Error in "+ylabels[i])
    if i == 2:
        ax1.set_yscale("log")
    ax1.legend()
    ax2.legend()    
    ax2.set_yscale("log")    
    plt.show()
    plt.show()
    

In [ ]:
# batch12 peak susceptibility scaling

print(Warr)
from scipy.optimize import curve_fit
def expdecay(x,gammanu,K):
    return K*x**gammanu
popt,pcov = curve_fit(expdecay,Larr,Warr[:,0])
print(popt)
print(np.sqrt(np.diag(pcov)))
for j in range(2,len(Larr)-2):
    if j == 2:
        plt.errorbar(Larr[j],Warr[j,0],yerr=1000*Warr[j,1],fmt="o",color="blue",label=r"Data, Errors$\times 1000$ ")
    else:
        plt.errorbar(Larr[j],Warr[j,0],yerr=1000*Warr[j,1],fmt="o",color="blue")

xarr = np.linspace(45,2200)

plt.plot(xarr,expdecay(xarr,*popt),label=fr"$\beta/\nu = ${-popt[0]:.4f}$\pm${np.sqrt(np.diag(pcov))[0]:.4f}")
# plt.hlines(popt[0],xmin=45,xmax=2156,color="black",label=r"$p_c$")
plt.xlabel(r"$L$")
plt.ylabel(r"$P(p_c,L)$")
plt.title("Finite Size Scaling of Critical Point Order Parameter")
plt.xscale("log")
plt.yscale("log")
plt.legend()